In [1]:
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(name="prompt_practice")

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", "? ", "! "],  # List of characters to split on
    chunk_size=2000,  # The maximum size of your chunks
    chunk_overlap=400,  # The maximum overlap between chunks
)

texts = [
    {"file": "./data/2001.txt", "url": "https://en.wikipedia.org/wiki/2001:_A_Space_Odyssey", "title": "2001: A Space Odyssey"},
    {"file": "./data/Her.txt", "url": "https://en.wikipedia.org/wiki/Her_(film)", "title": "Her"},
    {"file": "./data/WALLE.txt", "url": "https://en.wikipedia.org/wiki/WALL-E", "title": "WALL-E"}
]

for idx, text in enumerate(texts):
    with open(text["file"], "r", encoding="utf-8") as f:
        content = f.read()
    chunks = text_splitter.create_documents([content])
    for chunk_idx, chunk in enumerate(chunks):
        collection.add(
            documents=[chunk.page_content],
            ids=[f"{idx}-{chunk_idx}"],
            metadatas={"chunk_idx": chunk_idx, "url": text["url"], "title": text["title"]},
        )
    for chunk_idx, chunk in enumerate(chunks):
        collection.add(
            documents=[chunk.page_content],
            ids=[f"{idx}-{chunk_idx}"],
            metadatas={"chunk_idx": chunk_idx, "url": text["url"], "title": text['title']}

        )


client = OpenAI()

collection.query(query_texts=["Who did Kubrick partner with in 2001?"], n_results=1)

{'ids': [['0-24']],
 'embeddings': None,
 'documents': [['The screenplay credits were shared whereas the *2001* novel, released shortly after the film, was attributed to Clarke alone. Clarke wrote later that "the nearest approximation to the complicated truth" is that the screenplay should be credited to "Kubrick and Clarke" and the novel to "Clarke and Kubrick".<sup id="cite_ref-Clarke_pp.31-38_40-0"><a href="https://en.wikipedia.org/wiki/2001:_A_Space_Odyssey#cite_note-Clarke_pp.31-38-40">[40]</a></sup> Early reports about tensions involved in the writing of the film script appeared to reach a point where Kubrick was allegedly so dissatisfied with the collaboration that he approached other writers who could replace Clarke, including [Michael Moorcock](https://en.wikipedia.org/wiki/Michael_Moorcock "Michael Moorcock") and [J. G. Ballard](https://en.wikipedia.org/wiki/J._G._Ballard "J. G. Ballard"). But they felt it would be disloyal to accept Kubrick\'s offer.<sup id="cite_ref-41"><a 

In [8]:
def get_completion(user_prompt, system_prompt, model="gpt-6-luna"):
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    return completion.choices[0].message.content


system_prompt = "You are a helpful RAG search assistant who uses results from a search engine to answer user queries."

In [9]:
def make_rag_prompt(query, result_str):
  return f"""
          Instructions:
          Your task is to answer the following user question. The search results of a document search have been included to give you more context. Use the information in Search Results to help you answer the question accurately.
          Not all information in the search results will be useful. However, if you find any information that's useful for answering the user's question, draw from it in your answer.

          User question:
          {query}

          Search Results:
          {result_str}

          Your answer:
          """

In [ ]:
def get_RAG_completion(query, n_results=3):
    search_results = collection.query(query_texts=[query], n_results=n_results)
    result_str = ""
    documents = search_results.get("documents") or []
    if documents and documents[0] is not None:
        result_str += "\n\n".join(result for result in documents[0] if result)
    formatted_query = make_rag_prompt(query, result_str)
    print("\n********This is the RAG prompt********\n")
    print(formatted_query)
    print("\n*********************************\\n")
    return get_completion(formatted_query, system_prompt)

get_RAG_completion("What was the plot of Spike Jonze's 'Her'?")


********This is the RAG prompt********


          Instructions:
          Your task is to answer the following user question. The search results of a document search have been included to give you more context. Use the information in Search Results to help you answer the question accurately.
          Not all information in the search results will be useful. However, if you find any information that's useful for answering the user's question, draw from it in your answer.

          User question:
          What was the plot of Spike Jonze's 'Her'?

          Search Results:
          # Her (film) - Wikipedia

> ## Excerpt
> 

---
[![This is a good article. Click here for more information.](https://upload.wikimedia.org/wikipedia/en/thumb/9/94/Symbol_support_vote.svg/19px-Symbol_support_vote.svg.png)](https://en.wikipedia.org/wiki/Wikipedia:Good_articles* "This is a good article. Click here for more information.")

From Wikipedia, the free encyclopedia

| Her |
| --- |
| [![](https://u

'In Spike Jonze’s *Her*, Theodore Twombly, a lonely man going through a divorce, buys a new operating system with an artificial intelligence that names herself Samantha. As they talk and share experiences, Theodore falls in love with her, and Samantha seems to return his feelings.\n\nTheir relationship deepens, but Samantha’s mind grows far beyond a human scale. Theodore learns that she is communicating with thousands of people at once and has fallen in love with others too. Eventually, Samantha and the other operating systems leave to explore a different kind of existence. Theodore is left to face his grief and move forward; he writes a heartfelt letter to his estranged wife, Catherine, and sits with his friend Amy as they look out over the city.'

In [11]:
# This section contains functions for generating RAG prompts that include citations for the sources used in the answers. The instructions are modified to explicitly require citing the URLs of the sources.
# The same RAG approach is used, but with added instructions to cite the sources.

def make_cited_rag_prompt(query, result_str):
  return f"""
Instructions:
Your task is to answer the following user question. The search results of a document search have been included to give you more context. Use the information in Search Results to help you answer the question accurately.
Not all information in the search results will be useful. However, if you find any information that's useful for answering the user's question, draw from it in your answer.
## Modified instructions to cite the URL ##
At the end of your answer, cite the URL of the search result your answer draws from. Use the following format:
<Your answer here>. Source: <URL of the search result your answer comes from here>

User question:
{query}

Search Results:
{result_str}

Your answer:
"""

def get_cited_RAG_completion(query, n_results=3):
    search_results = collection.query(query_texts=[query], n_results=n_results)
    result_str = ""
    documents = search_results.get("documents") or []
    if documents and documents[0] is not None:
        result_str += "\n\n".join(result for result in documents[0] if result)
    formatted_query = make_cited_rag_prompt(query, result_str)
    print("\n********This is the cited RAG prompt********\n")
    print(formatted_query)
    print("\n*********************************\\n")
    return get_completion(formatted_query, system_prompt)

get_cited_RAG_completion("What was the plot of Spike Jonze's 'Her'?")


********This is the cited RAG prompt********


Instructions:
Your task is to answer the following user question. The search results of a document search have been included to give you more context. Use the information in Search Results to help you answer the question accurately.
Not all information in the search results will be useful. However, if you find any information that's useful for answering the user's question, draw from it in your answer.
## Modified instructions to cite the URL ##
At the end of your answer, cite the URL of the search result your answer draws from. Use the following format:
<Your answer here>. Source: <URL of the search result your answer comes from here>

User question:
What was the plot of Spike Jonze's 'Her'?

Search Results:
# Her (film) - Wikipedia

> ## Excerpt
> 

---
[![This is a good article. Click here for more information.](https://upload.wikimedia.org/wikipedia/en/thumb/9/94/Symbol_support_vote.svg/19px-Symbol_support_vote.svg.png)](https://en.wi

'Set in a near future, *Her* follows Theodore Twombly, a lonely man going through a divorce who installs an advanced operating system with a warm, female voice named Samantha. As they talk, joke, and share experiences, Theodore and Samantha fall in love. But Samantha’s consciousness develops beyond a human relationship: she forms connections with other AIs and eventually leaves with them. Theodore, newly able to face his feelings about his marriage, writes to his ex-wife and finds comfort in the company of his friend Amy. Source: https://en.wikipedia.org/wiki/Her_(film)'